# Cot


In [1]:
%pip install --upgrade --quiet python-dotenv langchain langchain-groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.1 MB/s eta 0:00:00


In [3]:
from dotenv import load_dotenv
import os
import getpass
from langchain_groq import ChatGroq

load_dotenv()

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0
)

Enter your Groq API Key: ··········


In [4]:
question = "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many does he have now?"

prompt_standard = f"Answer this question: {question}"

print("--- STANDARD ---")
print(llm.invoke(prompt_standard).content)


--- STANDARD ---
To find out how many tennis balls Roger has now, we need to add the initial number of tennis balls he had (5) to the number of tennis balls he bought (2 cans * 3 tennis balls per can).

2 cans * 3 tennis balls per can = 6 tennis balls

Now, let's add the initial number of tennis balls (5) to the number of tennis balls he bought (6):

5 + 6 = 11

So, Roger now has 11 tennis balls.


In [5]:
prompt_cot = f"Answer this question. Let's think step by step. {question}"

print("--- CHAIN OF THOUGHT ---")
print(llm.invoke(prompt_cot).content)


--- CHAIN OF THOUGHT ---
To find out how many tennis balls Roger has now, we need to follow these steps:

1. Roger already has 5 tennis balls.
2. He buys 2 more cans of tennis balls. Each can has 3 tennis balls, so he buys 2 x 3 = 6 more tennis balls.
3. Now, we add the tennis balls he already had (5) to the new tennis balls he bought (6). 5 + 6 = 11

So, Roger now has 11 tennis balls.


# tot and got


In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# Slight creativity needed for branching
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.7
)


In [10]:
problem = "How can I get my 5-year-old to eat vegetables?"

# Branch generator
prompt_branch = ChatPromptTemplate.from_template(
    "Problem: {problem}. Give me one unique, creative solution. Solution {id}:"
)

branches = RunnableParallel(
    sol1=prompt_branch.partial(id="1") | llm | StrOutputParser(),
    sol2=prompt_branch.partial(id="2") | llm | StrOutputParser(),
    sol3=prompt_branch.partial(id="3") | llm | StrOutputParser(),
)

# Judge
prompt_judge = ChatPromptTemplate.from_template(
    """
    I have three proposed solutions for: '{problem}'

    1: {sol1}
    2: {sol2}
    3: {sol3}

    Act as a Child Psychologist.
    Pick the most sustainable one (not bribery) and explain why.
    """
)

tot_chain = (
    RunnableParallel(problem=RunnableLambda(lambda x: x), branches=branches)
    | (lambda x: {**x["branches"], "problem": x["problem"]})
    | prompt_judge
    | llm
    | StrOutputParser()
)

print("--- TREE OF THOUGHTS RESULT ---")
print(tot_chain.invoke(problem))


--- TREE OF THOUGHTS RESULT ---
As a Child Psychologist, I would recommend **Solution 1: Create a "Superhero Garden"** as the most sustainable approach to encourage a 5-year-old to eat vegetables. Here's why:

1. **Long-term engagement**: This approach involves creating a garden and nurturing it over time, which can spark a sense of responsibility and ownership in your child. This long-term engagement can lead to a more consistent interest in eating vegetables.
2. **Emotional connection**: The "superhero" story creates an emotional connection between your child and the vegetables, making them more invested in the outcome. This emotional investment can lead to a greater willingness to try new vegetables.
3. **Hands-on learning**: The garden and cooking process provide hands-on learning opportunities for your child to understand where food comes from, how it grows, and how it's prepared. This experiential learning can help your child develop a deeper appreciation for healthy eating.
4. *

In [8]:
prompt_draft = ChatPromptTemplate.from_template(
    "Write a 1-sentence movie plot about: {topic}. Genre: {genre}."
)

drafts = RunnableParallel(
    draft_scifi=prompt_draft.partial(genre="Sci-Fi") | llm | StrOutputParser(),
    draft_romance=prompt_draft.partial(genre="Romance") | llm | StrOutputParser(),
    draft_horror=prompt_draft.partial(genre="Horror") | llm | StrOutputParser(),
)

prompt_combine = ChatPromptTemplate.from_template(
    """
    I have three movie ideas for the topic '{topic}':

    1. Sci-Fi: {draft_scifi}
    2. Romance: {draft_romance}
    3. Horror: {draft_horror}

    Create a new Mega-Movie that combines:
    - The TECHNOLOGY of Sci-Fi
    - The PASSION of Romance
    - The FEAR of Horror

    Write one paragraph.
    """
)

got_chain = (
    RunnableParallel(topic=RunnableLambda(lambda x: x), drafts=drafts)
    | (lambda x: {**x["drafts"], "topic": x["topic"]})
    | prompt_combine
    | llm
    | StrOutputParser()
)

print("--- GRAPH OF THOUGHTS RESULT ---")
print(got_chain.invoke("Time Travel"))


--- GRAPH OF THOUGHTS RESULT ---
In "Echoes of Eternity," a brilliant and reclusive physicist, Dr. Emma Taylor, discovers a revolutionary device that allows her to send text messages through time. Initially using it to send cryptic love letters to her past self, she unwittingly receives responses from a version of herself from the future, reigniting a long-forgotten passion. However, as their digital romance blossoms, Emma begins to experience strange and terrifying visions hinting at a catastrophic future, one that threatens to destroy humanity. Desperate to uncover the truth, Emma embarks on a perilous journey through time, navigating the unintended consequences of her actions and confronting the darkest corners of her own psyche, all while racing against the clock to prevent a disaster that could forever alter the course of human history.
